## 멜론 가사 수집 (장르별) 정적 스크래핑 

정적 수집시 좋아요는 화면 진입시 동적으로 생성되어 가져올 수 없는 버전 입니다. 

### 1. 환경 설정 

In [35]:
# 1. 필요 라이브러리 추가 
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
import os
from tqdm import tqdm
import time
import random

### 파라미터 세팅 함수 
#### url 특징 
 *  https://www.melon.com/genre/song_list.htm?gnrCode=GN0500
    * gnrCode = 장르별 코드 
    * GN0100 발라드 / GN0200 댄스 / GN0300 랩·힙합 / GN0400 R&b·Soul / GN0500 인디음악 / GN0600 록·메탈 / GN0700 트로트 / GN0800 포크·블루스 
    * GN0900 POP / GN1000 록·메탈 / GN1100 일렉트로니카 / GN1200 랩·힙합 / GN1300 R&b·Soul / GN1400 포크·블루스·컨트리
 * https://www.melon.com/song/detail.htm?songId=38427225
    * songId= 곡 ID 

In [38]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 장르 메뉴 정의
menu_abroad = {                 # 해외 장르 
    "록/메탈": "GN1000",
    "R&B/Soul": "GN1300"
}

menu_korea = {                  # 국내 장르 
    "발라드": "GN0100",
    "랩/힙합": "GN0300",
    "R&B/Soul": "GN0400",
    "인디음악": "GN0500",
    "트로트": "GN0600",
}

def genre_selection():
    # 반환할 딕셔너리 초기화
    ret_dict = {}

    # 국가 선택 위젯
    country_selector = widgets.Dropdown(
        options=[('국가를 선택해주십시오.', '0'), ('국내', '1'), ('국외', '2')],
        description='국가 선택:',
    )

    # 메뉴 선택 위젯, 초기에는 비어있음
    menu_selector = widgets.Dropdown(
        options=[],
        description='메뉴 선택:',
    )

    # 메뉴 출력 위젯
    menu_output = widgets.Output()

    def show_menu(change):
        """국가 선택 시 해당 메뉴를 보여주는 함수"""
        with menu_output:
            clear_output()  # 이전 출력을 지움
            if change['new'] == '1':
                menu = menu_korea
            elif change['new'] == '2':
                menu = menu_abroad
            else:
                menu = {}
                menu_selector.options = []
                return
            
            # 메뉴 옵션 업데이트
            menu_selector.options = list(menu.keys())
            print("메뉴가 업데이트되었습니다. 선택하세요.")
            menu_selector.layout.display = 'block'  # 메뉴 선택 레이아웃 표시

    # 메뉴 선택 후 해당 장르를 반환하는 함수
    def select_genre(change):
        nonlocal ret_dict  # 외부 변수에 접근
        selected_genre = change['new']
        if selected_genre:
            selected_value = list(menu_korea.values()) if country_selector.value == '1' else list(menu_abroad.values())
            index = menu_selector.options.index(selected_genre)
            ret_dict.update({
                "genre": selected_genre,
                "menu_key": selected_value[index]
            })
            #장르 클릭 
            
            with menu_output:
                clear_output()
            # 메뉴 선택 후 위젯 숨기기
            # menu_selector.layout.display = 'none'  # 메뉴 숨기기

    # 국가 선택 시 메뉴 보여주기
    country_selector.observe(show_menu, names='value')

    # 메뉴 선택 시 장르 출력
    menu_selector.observe(select_genre, names='value')

    # 위젯 표시
    display(country_selector, menu_output, menu_selector)

    # 장르 선택 결과가 나올 때까지 대기하는 대신, 결과를 반환
    return ret_dict

# 함수를 호출하여 실행
# result = genre_selection()
# print("선택된 장르 정보:", result)


Dropdown(description='국가 선택:', options=(('국가를 선택해주십시오.', '0'), ('국내', '1'), ('국외', '2')), value='0')

Output()

Dropdown(description='메뉴 선택:', options=(), value=None)

선택된 장르 정보: {}


### 2. 데이터 불러오기 

#### 2-1. 데이터 수집 화면 가져오기 

In [37]:

# # 사용자 입력을 받을 때까지 기다리는 방식
# selection_widget = genre_selection()

# # 사용자 입력이 들어올 때까지 대기
# while selection_widget is None:
#     pass  # 계속 대기

# # result = {'menu_key': selection_widget}  # 유저가 선택한 값을 딕셔너리에 저장
# # print(result)  # 정상 출력

Dropdown(description='국가 선택:', options=(('국가를 선택해주십시오.', '0'), ('국내', '1'), ('국외', '2')), value='0')

Output()

Dropdown(description='메뉴 선택:', options=(), value=None)

{'menu_key': {}}


In [ ]:
# request 를 사용하여 데이터 수집할 화면 가져오기 
headers = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                '(KHTML, like Gecko) Chrome/68.0.3440.75 Safari/537.36')
}

gnr_url = "https://www.melon.com/genre/song_list.htm"

#  장르 선택 함수 
genre_selection()

# 화면 여는 함수 
params = {
        'gnrCode': result['menu_key'],
    }

response = requests.get(gnr_url, params=params, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
song_list = soup.select('.wrap_song_info')


In [ ]:
# 데이터프레임 초기화
# columns = ['chartDate', 'rank', 'title', 'singer', 'album_name', 'release_date', 'genre', 'lyric', 'composer', 'lyricist', 'arranger']
# 곡 제목 title
# 가사 lylics
# 아티스트 artist
# 장르 ganre
# 발매일 date
# 좋아요 like
columns = ['title', 'artist', 'ganre', 'release_date', 'like_cnt', 'lylics']
song_data = pd.DataFrame(columns=columns)

# tqdm 라이브러리로 진행 상황 바 표시
for i, meta in tqdm(enumerate(song_list, 1), total=len(song_list), desc="Processing songs"):
    rank = i

    # 모든 곡 정보를 포함하는 요소 선택
    songs = meta.select('.wrap_song_info')

    # 각 곡 정보에서 곡 제목 추출
    for song in song_list:
        title_element = song.select_one('.ellipsis.rank01 a')  # 곡 제목 선택
        if title_element:  # 요소가 존재할 경우
            # song_titles.append(title_element.text.strip())  # 제목을 리스트에 추가
            title = title_element.text.strip()
            href = title_element['href']  # href 속성 가져오기
            # 정규 표현식을 사용하여 곡 ID 추출
            match = re.search(r"playSong\('(\d+)',(\d+)\)", href)
            # if match:
            song_id = match.group(2)  # 두 번째 그룹이 곡 ID
            song_url = 'https://www.melon.com/song/detail.htm?songId=' + song_id

            response = requests.get(song_url, params=params, headers=headers)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 가수
            singer_html = soup.select('.wrap_info .artist a')
            singer_s = ', '.join([html['title'] for html in singer_html if html['title']]) if singer_html else 'Various Artists'

            # 앨범명
            # album_name = soup.select('.list dd')[0].get_text(strip=True)

            # 발매날짜
            release_date = soup.select('.list dd')[1].get_text(strip=True)

            # 장르
            genre = soup.select('.list dd')[2].get_text(strip=True)

            # 좋아요 
            # <span id="d_like_count" class="cnt">44</span>
            # like_cnt = soup.select('.cnt').get_text(strip=True)

            # 예시 코드
            like_count_element = meta.select_one('#d_like_count')  # ID로 요소 선택
            if like_count_element:  # 요소가 존재할 경우
                like_cnt = like_count_element.text.strip()  # 텍스트 가져오기 및 공백 제거
            else:
                like_cnt = 0


            # 가사
            lyric = '없음'
            lyric_html = soup.select_one('.section_lyric .wrap_lyric .lyric')
            if lyric_html:
                lyric = lyric_html.get_text(strip=True, separator='\n')

            row = pd.Series([title, singer_s, genre, release_date, like_cnt,  lyric], index=song_data.columns)
            song_data = pd.concat([song_data, pd.DataFrame([row])], ignore_index=True)

            # 1초에서 5초 사이의 랜덤한 시간 선택
            random_sleep_time = random.uniform(1, 5)
            time.sleep(random_sleep_time)  # IP 차단 방지용 랜덤한 시간 동안 대기